<div dir="rtl" lang="he" style="text-align: right;">

# מפגש 17 — KNN בספרייה: אימון וחיזוי
2 שעות הוראה ותרגול ו־1 שעות פרויקט; יחד 135 דקות.

## סיפור המסגרת
נפעיל KNN על מדידות אמיתיות מ־Penguins ונבדוק לא רק כמה צדק, אלא גם באילו דוגמאות טעה.

## מטרות
- להפעיל fit ו־predict.
- לחשב Accuracy.
- לנתח טעות.

</div>

<div dir="rtl" lang="he" style="text-align: right;">

## הנחיות למורה
זהו פתרון מתועד למעבדה. מבדילים בין תוצאה נכונה לבין נימוק נכון, ובודקים שהקוד תואם לנתוני התלמיד. המקורות הישנים נשמרו ללא שינוי.

</div>

<div dir="rtl" lang="he" style="text-align: right;">

## פתרון חלק 1
הפתרון הבא תואם לנתוני התרגול העצמאי, ולא לנתוני הדוגמה הפתורה.

</div>

<div dir="rtl" lang="he" style="text-align: right;">

פתרון לתרגול העצמאי. תשובות מילוליות חלופיות מתקבלות אם הן מנומקות. התוצאות המספריות נקבעות בהרצה מול גרסת הנתונים הנוכחית.

</div>

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

url = "https://raw.githubusercontent.com/samiabo75/Data-Science-ML-Course/main/Datasets/penguins.csv"
df = pd.read_csv(url)
display(df.head())
display(df.isna().sum())

<div dir="rtl" lang="he" style="text-align: right;">

## 1. קלט, יעד וחלוקה
בחרו bill_length_mm ו־ bill_depth_mm בלבד, ו־ species כיעד. הסירו רק שורות ללא יעד בעותק עבודה. חלקו יחד, test_size=0.25, random_state=17, stratify=y. לפני ההרצה הסבירו מדוע היעד אינו מאפיין.

המאפיינים הם שתי מדידות מקור במ״מ, והיעד הוא מין. הכנסת היעד לקלט תחשוף למודל את התשובה. חלוקה משותפת שומרת התאמה בין שורות לתוויות.

</div>

In [ ]:
model_df = df.dropna(subset=["species"]).copy()
features = ["bill_length_mm","bill_depth_mm"]
X = model_df[features].copy()
y = model_df["species"].copy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)
print("Train:", len(X_train), "Test:", len(X_test))

<div dir="rtl" lang="he" style="text-align: right;">

## 2. הכנה ללא זליגה
חשבו ממוצעים מאימון והשלימו בשתי הקבוצות לפיהם. התאימו StandardScaler לאימון המושלם בלבד והמירו את שתי הקבוצות. הסבירו מדוע אין fit נוסף לבדיקה.

הממוצעים ופרמטרי התקנון נלמדים מאימון. שימוש ב־ Test לחישובם יאפשר למידע מההערכה להשתתף בהכנה.

</div>

In [ ]:
training_means = X_train.mean()
X_train_ready = X_train.fillna(training_means)
X_test_ready = X_test.fillna(training_means)

scaler = StandardScaler().set_output(transform="pandas")
scaler.fit(X_train_ready)
X_train_scaled = scaler.transform(X_train_ready)
X_test_scaled = scaler.transform(X_test_ready)
display(X_train_scaled.head())

<div dir="rtl" lang="he" style="text-align: right;">

## 3. אימון ותחזית
הגדירו מראש K=3, weights='uniform', metric='euclidean'. הפעילו fit על האימון המתוקנן והתוויות, ו־ predict על הבדיקה המתוקננת. ודאו שמספר התחזיות שווה למספר רשומות הבדיקה.

fit מקבל X_train_scaled ו־ y_train. predict מקבל X_test_scaled בלבד. K קבוע מראש; אין כאן תהליך בחירת K.

</div>

In [ ]:
model = KNeighborsClassifier(n_neighbors=3, weights="uniform", metric="euclidean")
model.fit(X_train_scaled, y_train)
predictions = model.predict(X_test_scaled)
print(predictions[:5])

<div dir="rtl" lang="he" style="text-align: right;">

## 4. הערכה
צרו comparison עם actual ו־ predicted ובאינדקס y_test. הוסיפו correct, ספרו נכונות וחשבו Accuracy ביד וב־ accuracy_score. דווחו שבר, יחס ואחוז.

הטבלה משווה תוויות, לא מדידות. correct הוא תנאי לכל שורה; סכומו הוא מספר ההצלחות. יחס זה צריך להסכים ל־ accuracy_score.

</div>

In [ ]:
comparison = pd.DataFrame(
    {"actual": y_test, "predicted": predictions}, index=y_test.index
)
comparison["correct"] = comparison["actual"] == comparison["predicted"]
display(comparison.head(10))
correct_count = comparison["correct"].sum()
accuracy = accuracy_score(y_test, predictions)
print("Correct:", correct_count, "out of", len(y_test))
print("Accuracy:", accuracy, "Percent:", 100 * accuracy)

<div dir="rtl" lang="he" style="text-align: right;">

## 5. ניתוח טעויות
סננו שגיאות והשוו actual ל־ predicted בעד שלוש רשומות. אם אין טעויות, ציינו זאת. כתבו מגבלה אחת של החלוקה ואחת של המדד. אין לכוון K לפי התוצאות.

החלפות בין מינים הן טעויות סיווג. גם Accuracy גבוה אינו מציג מי נפגע מכל טעות, או מה יקרה באוכלוסייה חדשה. אין להסיק שכל הטעויות נגרמו ממרחק דומה בלי לבדוק שכנים.

</div>

In [ ]:
errors = comparison[comparison["correct"] == False]
print("Errors:", len(errors))
display(errors.head(10))

In [ ]:
assert len(predictions) == len(y_test)
assert abs(accuracy - correct_count / len(y_test)) < 1e-12

<div dir="rtl" lang="he" style="text-align: right;">

## 6. כרטיס מודל לדוגמה
שאלה: האם ניתן לסווג מין פינגווין משתי מדידות מקור? רשומה היא פינגווין נמדד; מקור Penguins וייחוס כפי שתועדו לעיל. X: אורך ועומק מקור, y: species. חלוקה 75%/25%, stratify לפי יעד, seed=17. ללא יעד אין דוגמה מפוקחת; חסרי מדידות הושלמו בממוצעי אימון; התקנון הותאם לאימון בלבד. KNN אוקלידי, K=3 ומשקל שווה. המדד והטעויות מצוטטים מפלט ההרצה, ולא ממספר משוער.
מגבלות: מדגם לימודי, חלוקה אחת ושני מאפיינים בלבד; אין הוכחת הכללה לשטח. Accuracy אינו מפרט ביצוע לכל מין. אין להסיק מהבדל מול המחברת המודרכת על עדיפות מאפיינים בגלל חלוקה שונה.
בפרויקט אמיתי נצטרך להגדיר שימוש, מקור ורישוי, מדד המתאים למחיר הטעות ותהליך אימות ובחירת פרמטרים שאינו משתמש ב־ Test.

</div>

<div dir="rtl" lang="he" style="text-align: right;">

## תשובה לכרטיס היציאה
לא. הוא מקבל X בלבד. האמת נדרשת לאחר החיזוי לצורך הערכה.

</div>

<div dir="rtl" lang="he" style="text-align: right;">

## משוב דידקטי
בדקו בנפרד: בחירת פעולה, תקינות קוד, פירוש פלט, יחידות ומגבלה. שאלו מה ישתנה בקלט חדש. אם יש קושי, חזרו לדוגמה קטנה לפני הגדלת המאגר. זהו משוב מקומי, לא מחוון ניקוד רשמי.

</div>